# 08b. Identity diagnostics and incremental value

- A. Identity probe (D1): site/sensor prediction from the spectrum, within and across classes
- B. Sensor-level label permutation (D1): labels as a pure function of sensor identity
- C. Incremental value (D1, D2, D3): metadata only / acoustic only / combined, identical folds
- Results are diagnostic, not causal shares

In [2]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, evaluation as ev, sfb, diagnostics as dg, reporting as rp
cfg.apply_mpl_style(); cfg.banner()
import warnings; warnings.filterwarnings("ignore")

X1, X1_max, M1, HZ = io_utils.load_d1()
y5 = M1.cls.to_numpy(); site = M1.site.to_numpy(); sid = M1.sid.to_numpy()
dt = pd.to_datetime(M1.dt).to_numpy()
CAP = cfg.D1_SUBSAMPLE

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000


## 1. Identity probe (D1)

- Model: random forest; random 70/30 row split
- Targets: site (all, within normal, within out), sensor (within normal, within out)
- IDs with ≥ 20 (site) or ≥ 40 (sensor) rows; equal rows per ID, max 30,000 total
- Reference: majority-ID rate (chance)

In [3]:
probe_rows = []
specs = [("site", site, None, 20), ("site", site, "normal", 20), ("site", site, "out", 20),
         ("sensor", sid, "normal", 40), ("sensor", sid, "out", 40)]
for what, ids, within, min_rows in specs:
    m = None if within is None else (y5 == within)
    r = dg.identity_probe(X1, ids, min_rows=min_rows, kind="rf", seed=0, mask=m, max_rows=CAP)
    probe_rows.append(dict(identifier=what, within_class=within or "all", min_rows=min_rows, **r))
    print(probe_rows[-1], flush=True)
probe = pd.DataFrame(probe_rows)
probe.to_csv(cfg.RESULTS / "E12_identity_probe.csv", index=False)
print(probe.round(4).to_string(index=False))
for _, r in probe.iterrows():
    rp.record(f"D1.probe.{r.identifier}.{r.within_class}.acc", r.acc, "E12_identity_probe.csv", "R2-1",
              note=f"chance={r.chance:.4f}, n_ids={r.n_ids}")

{'identifier': 'site', 'within_class': 'all', 'min_rows': 20, 'n_ids': 43, 'n_rows': 18553, 'acc': 0.7853036291771469, 'chance': 0.037549407114624504, 'uniform_chance': 0.023255813953488372, 'lift_over_chance': 20.913875598086126}
{'identifier': 'site', 'within_class': 'normal', 'min_rows': 20, 'n_ids': 7, 'n_rows': 22152, 'acc': 0.7454107733975324, 'chance': 0.4819440264820945, 'uniform_chance': 0.14285714285714285, 'lift_over_chance': 1.5466749921948173}
{'identifier': 'site', 'within_class': 'out', 'min_rows': 20, 'n_ids': 32, 'n_rows': 19719, 'acc': 0.9420216362407032, 'chance': 0.14874915483434753, 'uniform_chance': 0.03125, 'lift_over_chance': 6.3329545454545455}
{'identifier': 'sensor', 'within_class': 'normal', 'min_rows': 40, 'n_ids': 182, 'n_rows': 11265, 'acc': 0.8494082840236686, 'chance': 0.013609467455621301, 'uniform_chance': 0.005494505494505495, 'lift_over_chance': 62.413043478260875}
{'identifier': 'sensor', 'within_class': 'out', 'min_rows': 40, 'n_ids': 221, 'n_rows

## 2. Sensor-level label permutation (D1)

- Each sensor assigned the label of another randomly drawn sensor; spectra unchanged
- Protocols: P1 naive, P2 sensor-independent, P3 site-independent
- GBDT, 5 replicates, training rows capped at 30,000; true labels as reference

In [ ]:
PERM = cfg.RESULTS / "E12_permutation_control.csv"
if PERM.exists():
    pc = pd.read_csv(PERM)
else:
    pc = dg.permutation_control(X1, y5, site, sid, dt,
                                ["P1_naive_random", "P2_sensor_indep", "P3_site_indep"],
                                n_perm=cfg.N_PERM_REFIT, kind="gbdt", subsample=CAP)
    pc.to_csv(PERM, index=False)
pcs = pc.groupby(["labels", "protocol"]).agg(acc=("acc", "mean"), acc_sd=("acc", "std"),
                                             bacc=("bacc", "mean"), base=("base", "mean"),
                                             excess=("excess", "mean"), n_reps=("acc", "size"),
                                             relabelled=("frac_rows_relabelled", "mean")).reset_index()
pcs.to_csv(cfg.RESULTS / "E12_permutation_control_summary.csv", index=False)
print(pcs.round(4).to_string(index=False))
for _, r in pcs.iterrows():
    rp.record(f"D1.perm.{r.labels}.{r.protocol}.acc", r.acc, "E12_permutation_control_summary.csv", "R2-1",
              note=f"base={r.base:.4f}, bacc={r.bacc:.4f}, reps={int(r.n_reps)}")

# P1 excess with permuted labels / P1 excess with true labels
t = pcs.set_index(["labels", "protocol"])
if ("sensor_permuted", "P1_naive_random") in t.index and ("true", "P1_naive_random") in t.index:
    ratio = t.loc[("sensor_permuted", "P1_naive_random"), "excess"] / t.loc[("true", "P1_naive_random"), "excess"]
    print(f"\nexcess over majority under P1 with sensor-permuted labels, as a fraction of the true-label "
          f"excess: {ratio:.3f}")
    rp.record("D1.perm.P1.excess_ratio_perm_over_true", ratio, "E12_permutation_control_summary.csv", "R2-1",
              note="interpretive: identity-only attainable share of the naive excess")

         labels        protocol    acc  acc_sd   bacc   base  excess  n_reps  relabelled
sensor_permuted P1_naive_random 0.7591  0.0057 0.6526 0.4605  0.2986       5       0.743
sensor_permuted P2_sensor_indep 0.3854  0.0188 0.1987 0.4601 -0.0747       5       0.743
sensor_permuted   P3_site_indep 0.3794  0.0142 0.1998 0.4572 -0.0778       5       0.743
           true P1_naive_random 0.8932  0.0012 0.8563 0.3149  0.5782       5       0.000
           true P2_sensor_indep 0.5435  0.0183 0.4232 0.3203  0.2232       5       0.000
           true   P3_site_indep 0.4491  0.0988 0.3698 0.3787  0.0705       5       0.000

excess over majority under P1 with sensor-permuted labels, as a fraction of the true-label excess: 0.516


## 3. Incremental value: D1

- Inputs: `lrate` + `llevel` / spectrum (513 bins) / both
- Tasks: five-class, leak vs. non-leak, alarm vs. normal, outdoor vs. indoor
- Folds: StratifiedGroupKFold by site (5 folds, OOF); training rows capped at 30,000
- GBDT; intervals: site-level bootstrap

In [5]:
INC1 = cfg.RESULTS / "E12_incremental_D1.csv"
tasks = {
    "T5_five_class": (np.ones(len(y5), bool), y5),
    "TL_leak_vs_nonleak": (np.ones(len(y5), bool), np.where(np.isin(y5, ["out", "in"]), "leak", "nonleak")),
    "TA_alarm_vs_normal": (np.ones(len(y5), bool), np.where(y5 == "normal", "normal", "alarm")),
    "TR_out_vs_in": (np.isin(y5, ["out", "in"]), y5),
}
dev = M1[["lrate", "llevel"]].to_numpy(float)
if INC1.exists():
    inc1 = pd.read_csv(INC1)
else:
    parts = []
    for t, (mask, yy) in tasks.items():
        ys = yy[mask].astype(str)
        folds = ev.make_folds("stratified_group", ys, site[mask], 5, seed=0)
        sets = {"metadata: lrate + llevel": dev[mask], "acoustic: 513 bins": X1[mask],
                "acoustic + lrate + llevel": np.hstack([X1[mask], dev[mask]])}
        summ, _ = dg.incremental_value(sets, ys, folds, kind="gbdt", seed=0, clusters=site[mask],
                                       subsample=CAP, n_boot=cfg.N_BOOT)
        summ.insert(0, "task", t)
        parts.append(summ)
        print(summ.drop(columns="cm").round(4).to_string(index=False), flush=True)
    inc1 = pd.concat(parts, ignore_index=True)
    inc1.insert(0, "dataset", "D1")
    inc1["protocol"] = "site-grouped 5-fold OOF"
    inc1.to_csv(INC1, index=False)
print(inc1.drop(columns="cm").round(4).to_string(index=False))
for _, r in inc1.iterrows():
    rp.record(f"D1.incremental.{r.task}.{r.feature_set}.acc", r.acc, "E12_incremental_D1.csv", "R2-3,R2-4",
              lo=r.get("acc_lo"), hi=r.get("acc_hi"), note=f"bacc={r.bacc:.4f}; unit=site")

         task               feature_set     n  n_features    acc   bacc    mf1  fold_majority_rule  excess_over_rule  acc_lo  acc_hi  excess_lo  excess_hi  n_clusters
T5_five_class  metadata: lrate + llevel 70384           2 0.5829 0.4068 0.3709              0.2223            0.3606  0.4208  0.7297     0.1667     0.5545          49
T5_five_class        acoustic: 513 bins 70384         513 0.5202 0.3941 0.3892              0.2223            0.2979  0.3680  0.6590     0.1038     0.4813          49
T5_five_class acoustic + lrate + llevel 70384         515 0.5643 0.4303 0.4288              0.2223            0.3420  0.3982  0.7171     0.1422     0.5384          49
              task               feature_set     n  n_features    acc   bacc    mf1  fold_majority_rule  excess_over_rule  acc_lo  acc_hi  excess_lo  excess_hi  n_clusters
TL_leak_vs_nonleak  metadata: lrate + llevel 70384           2 0.8074 0.8102 0.8015              0.4285            0.3789  0.7043  0.8872     0.1356     0.5925 

## 4. Incremental value: D3

- Inputs: metadata (topology, flow, noise, channel; leak type excluded) / acoustic (199 features) / both
- Folds: run-independent (5 folds) and operating-condition LOCO (6 folds), OOF over 3,600 segments
- GBDT; intervals: physical-run bootstrap

In [6]:
M3 = io_utils.scan_d3()
H = M3[(~M3.is_bg) & (M3.modality == "hydrophone") & M3.label.notna()].reset_index(drop=True)
c = io_utils.load_cache(f"d3_seg_hydrophone_{cfg.WIN_SEC}_pcm32", H.file_id, by_unique=True)
assert c is not None, "run notebook 06 first"
S = H.set_index("file_id").loc[c["ids"].astype(str)].reset_index()
y3 = S.label.to_numpy().astype(str)
meta3 = np.column_stack([pd.Categorical(S[col].astype(str)).codes for col in ["topology", "flow", "noise", "sensor"]]).astype(float)
parts = []
for pname, scheme, grp in [("run-independent 5-fold OOF", "stratified_group", S.run_id.to_numpy()),
                           ("operating-condition LOCO OOF", "logo", S.op_condition.to_numpy())]:
    folds = ev.make_folds(scheme, y3, grp, 5, seed=0)
    summ, _ = dg.incremental_value({"metadata: topology, flow, noise, channel": meta3,
                                    "acoustic: 199 mel features": c["X"],
                                    "acoustic + metadata": np.hstack([c["X"], meta3])},
                                   y3, folds, kind="gbdt", seed=0, clusters=S.run_id.to_numpy(),
                                   n_boot=cfg.N_BOOT)
    summ.insert(0, "protocol", pname)
    parts.append(summ)
inc3 = pd.concat(parts, ignore_index=True)
inc3.insert(0, "task", "leak vs no-leak"); inc3.insert(0, "dataset", "D3")
inc3.to_csv(cfg.RESULTS / "E12_incremental_D3.csv", index=False)
print(inc3.drop(columns="cm").round(4).to_string(index=False))
for _, r in inc3.iterrows():
    rp.record(f"D3.incremental.{r.protocol}.{r.feature_set}.acc", r.acc, "E12_incremental_D3.csv", "R2-3",
              lo=r.acc_lo, hi=r.acc_hi, note=f"bacc={r.bacc:.4f}; unit=run")

dataset            task                     protocol                              feature_set    n  n_features    acc   bacc    mf1  fold_majority_rule  excess_over_rule  acc_lo  acc_hi  excess_lo  excess_hi  n_clusters
     D3 leak vs no-leak   run-independent 5-fold OOF metadata: topology, flow, noise, channel 3600           4 0.8000 0.5000 0.4444                 0.8            0.0000  0.7000  0.9000     0.0000     0.0000          60
     D3 leak vs no-leak   run-independent 5-fold OOF               acoustic: 199 mel features 3600         199 0.9358 0.8604 0.8909                 0.8            0.1358  0.8997  0.9686     0.0622     0.2156          60
     D3 leak vs no-leak   run-independent 5-fold OOF                      acoustic + metadata 3600         203 0.9019 0.7793 0.8212                 0.8            0.1019  0.8456  0.9517     0.0364     0.1725          60
     D3 leak vs no-leak operating-condition LOCO OOF metadata: topology, flow, noise, channel 3600           4 0.8000 0.

## 5. Incremental value: D2 (from notebook 07) and combined table

In [7]:
p = cfg.RESULTS / "E7_acoustic_vs_clock.csv"
if p.exists():
    avc = pd.read_csv(p)
    inc2 = avc.rename(columns={"predictor": "feature_set", "lo_seed0": "acc_lo", "hi_seed0": "acc_hi"})
    inc2.insert(0, "dataset", "D2"); inc2["task"] = "leak vs no-leak"
    inc2["protocol"] = "logger-grouped 5-fold OOF (" + inc2.population + ")"
    cols = ["dataset", "task", "protocol", "feature_set", "n", "acc", "acc_lo", "acc_hi", "bacc", "rule"]
    allinc = pd.concat([inc1.rename(columns={"fold_majority_rule": "rule"})[cols],
                        inc2[cols],
                        inc3.rename(columns={"fold_majority_rule": "rule"})[cols]], ignore_index=True)
    allinc.to_csv(cfg.RESULTS / "E12_incremental_all.csv", index=False)
    print(allinc.round(4).to_string(index=False))
else:
    print("run notebook 07 first")

dataset               task                                    protocol                              feature_set     n    acc  acc_lo  acc_hi   bacc   rule
     D1      T5_five_class                     site-grouped 5-fold OOF                 metadata: lrate + llevel 70384 0.5829  0.4208  0.7297 0.4068 0.2223
     D1      T5_five_class                     site-grouped 5-fold OOF                       acoustic: 513 bins 70384 0.5202  0.3680  0.6590 0.3941 0.2223
     D1      T5_five_class                     site-grouped 5-fold OOF                acoustic + lrate + llevel 70384 0.5643  0.3982  0.7171 0.4303 0.2223
     D1 TL_leak_vs_nonleak                     site-grouped 5-fold OOF                 metadata: lrate + llevel 70384 0.8074  0.7043  0.8872 0.8102 0.4285
     D1 TL_leak_vs_nonleak                     site-grouped 5-fold OOF                       acoustic: 513 bins 70384 0.7446  0.6531  0.8211 0.7450 0.4285
     D1 TL_leak_vs_nonleak                     site-grouped 5-fold OOF